# 02 — How Similar Are Packed Blocks?

We pack the shared example grids into crank words, then ask Crankl a few “how connected / how similar?” questions.

**Picture it simply:**

- Each packed word is a **slot** in a short chain.
- Neighbors in that chain talk to each other.
- Crankl reports rough health checks — not a full topology thesis.

What you get:

- **`beta1_proxy`** — a cheap count of “loops” in the neighbor graph. With only two slots in a line, expect `0` (a line has no loop).
- **`sheaf_resonance`** — one number saying how similar two packed archives look, using both local word similarity and neighbor patterns.

These are **proxies** (quick diagnostics), not a complete mathematical cohomology calculation.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np

notebook_dir = Path.cwd() / "notebooks" if (Path.cwd() / "notebooks").exists() else Path.cwd()
sys.path.insert(0, str(notebook_dir)) if str(notebook_dir) not in sys.path else None

from crankl_demo import CranklAPI, prepare_demo_files

api = CranklAPI()
demo = prepare_demo_files()

# Packing converts each 8x8 block into one neighboring slot in the sequence.
source_words = api.pack(demo.source_blocks)
target_words = api.pack(demo.target_blocks)

print("Source crank words:", [f"0x{word:016x}" for word in source_words])
print("Target crank words:", [f"0x{word:016x}" for word in target_words])
print("Source archive metrics:")
print(json.dumps(api.metrics(source_words), indent=2))

## Topology score and sequence similarity

- Run `beta1_proxy` on source and on target separately.
- Run `sheaf_resonance` to compare the two sequences as wholes.
- Also print per-slot Clifford resonance so you can see local similarity slot-by-slot.

With only two adjacent slots, the loop count is usually **0**. That is correct: two dots connected by one edge cannot form a loop.

In [ ]:
source_beta1 = api.beta1_proxy(source_words)
target_beta1 = api.beta1_proxy(target_words)
sheaf_similarity = api.sheaf_resonance(source_words, target_words)
local_clifford = [
    api.resonance(int(source), int(target))
    for source, target in zip(source_words, target_words)
]

print(f"Source beta1 proxy: {source_beta1}")
print(f"Target beta1 proxy: {target_beta1}")
print(f"Sheaf resonance (proxy): {sheaf_similarity:.6f}")
print("Per-slot Clifford resonance:", np.round(local_clifford, 6))

## How to read the numbers

- **`beta1_proxy = 0`** — no loop found in the neighbor graph. That does *not* mean the data is empty or boring.
- **Per-slot Clifford resonance** — local “are these two words alike?” scores.
- **Sheaf resonance** — also looks at neighbor patterns, so it may differ from a plain average of the local scores.

The public API does not expose the internal neighbor weights or a full coboundary matrix. Treat these as diagnostic meters.

In [ ]:
assert source_beta1 >= 0 and target_beta1 >= 0
assert np.isfinite(sheaf_similarity)
assert all(np.isfinite(local_clifford))
print("All topology proxy outputs are finite and the beta1 proxies are non-negative.")